In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import os
from datetime import datetime

In [ ]:

valid_methods = [
    "variance",
    "correlation",
    "chi_squared",
    "mutual_information",
    "anova_f_test",
]
n_features = 50

data_name = "Lymphoma"
data_dir = f"data/processed/{data_name}"
all_selected_features = []

In [ ]:
print("󰈞 Collect filtered features from 5 methods...\n")

for m in valid_methods:
    data_path = f"{data_dir}/{data_name}_{m}_{n_features}features.csv"

    try: 
        # nrows = 0 -> take only coll name, not data
        df_cols = pd.read_csv(data_path, nrows=0)

        # take features (from index 1 -> end)
        features = df_cols.columns[1:].tolist()

        # put it in all_selected_features
        all_selected_features.extend(features)
    except FileNotFoundError:
        print(f" Error: Not found file {data_path}")

print("󰄬 Done")

In [ ]:
# counting (vote) -> the most gen is selected
feature_count = Counter(all_selected_features)

df_counts = pd.DataFrame.from_dict(feature_count, orient='index', columns=['Votes']).reset_index()
df_counts.rename(columns={'index': 'Feature'}, inplace=True)

# sort with votes
df_counts = df_counts.sort_values(by='Votes', ascending=False).reset_index(drop=True)

In [ ]:
# display top 10 gen
df_top10 = df_counts.head(10)
display(df_top10) #type: ignore




In [ ]:
# setup saving
report_dir = f"result/{data_name}/report"
plot_dir = f"result/{data_name}/plot"
csv_dir = f"data/processed/{data_name}"
os.makedirs(report_dir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)

In [ ]:

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
report_path = f"{report_dir}/top{n_features}_features_voting_{timestamp}.txt"
plot_path = f"{plot_dir}/top{n_features}_features_voting_{timestamp}.png"
csv_path = f"{csv_dir}/top{n_features}_features_voting_{timestamp}.csv"

In [ ]:
# 
top_n_plot = 10
df_plot = df_counts.head(top_n_plot)

plt.figure(figsize=(10, 8))
sns.barplot(data=df_plot, x='Votes', y='Feature',hue='Feature', palette='magma')

plt.title(f"Top {top_n_plot} Gen:")
# plt.xlabel('Votes (max = 5)')
# plt.ylabel('Gen (feature)', fontweight='bold')
#
plt.xticks(range(1,6))
# plt.grid(axis='x', linestyle='--', alpha=0.7)
#
# plt.tight_layout()
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"󰋩 Saved plot in {plot_path}")
plt.show()

In [ ]:
with open(report_path, "w", encoding="utf-8") as f:
    f.write("Ensemble filter selection report \n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Number of method: {len(valid_methods)}\n")
    f.write("-"*60+"\n")

    f.write(df_top10.to_string(index=False))

print(f"󰎞 saved report in {report_path}")
df_top10.to_csv(csv_path, index=False)
print(f"󰈙 Saved CSV data in {csv_path}")